# Sauvegarde et Chargement de Modèles

Dans ce notebook, je vais te montrer comment sauvegarder et charger des modèles avec PyTorch. C’est important car tu voudras souvent recharger des modèles déjà entraînés pour faire des prédictions ou pour continuer l’entraînement sur de nouvelles données.


In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt

import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
from torchvision import datasets, transforms

import helper
import fc_model

In [2]:
# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])
# Download and load the training data
trainset = datasets.FashionMNIST('~/.pytorch/F_MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Download and load the test data
testset = datasets.FashionMNIST('~/.pytorch/F_MNIST_data/', download=True, train=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=True)

Voici l’une des images.

In [3]:
image, label = next(iter(trainloader))
helper.imshow(image[0,:]);

: 

# Entraîner un réseau

Pour rendre les choses plus concises ici, j’ai déplacé l’architecture du modèle et le code d’entraînement de la partie précédente dans un fichier appelé `fc_model`. En l’important, nous pouvons facilement créer un réseau entièrement connecté avec `fc_model.Network`, et entraîner le réseau en utilisant `fc_model.train`. J’utiliserai ce modèle (une fois entraîné) pour montrer comment sauvegarder et charger des modèles.


In [ ]:
# Create the network, define the criterion and optimizer

model = fc_model.Network(784, 10, [512, 256, 128])
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
fc_model.train(model, trainloader, testloader, criterion, optimizer, epochs=2)

NameError: name 'trainloader' is not defined

## Sauvegarde et chargement de réseaux

Comme tu peux l’imaginer, il n’est pas pratique de réentraîner un réseau à chaque fois que tu veux l’utiliser. À la place, nous pouvons sauvegarder les réseaux déjà entraînés, puis les recharger plus tard pour continuer l’entraînement ou les utiliser pour faire des prédictions.

Les paramètres des réseaux PyTorch sont stockés dans le `state_dict` d’un modèle. On peut voir que le *state dict* contient les matrices de poids et de biais pour chacune de nos couches.


In [ ]:
print("Our model: \n\n", model, '\n')
print("The state dict keys: \n\n", model.state_dict().keys())

La chose la plus simple à faire est de simplement sauvegarder le *state dict* avec `torch.save`. Par exemple, nous pouvons le sauvegarder dans un fichier `'checkpoint.pth'`. 


In [ ]:
torch.save(model.state_dict(), 'checkpoint.pth')

 Ensuite, nous pouvons charger le *state dict* avec `torch.load`. 


In [ ]:
state_dict = torch.load('checkpoint.pth')
print(state_dict.keys())

Et pour charger le *state dict* dans le réseau, il suffit d’utiliser `model.load_state_dict(state_dict)`.


In [ ]:
model.load_state_dict(state_dict)

Cela paraît simple, mais comme souvent c’est un peu plus compliqué. Le chargement du *state dict* ne fonctionne que si l’architecture du modèle est exactement la même que celle du point de contrôle. Si je crée un modèle avec une architecture différente, cela échoue.


In [ ]:
# Try this
model = fc_model.Network(784, 10, [400, 200, 100])
# This will throw an error because the tensor sizes are wrong!
model.load_state_dict(state_dict)

Cela signifie que nous devons reconstruire le modèle exactement tel qu’il était lors de l’entraînement. Les informations sur l’architecture du modèle doivent être sauvegardées dans le point de contrôle, en plus du *state dict*. Pour ce faire, on construit un dictionnaire contenant toutes les informations nécessaires pour reconstruire complètement le modèle. 


In [ ]:
checkpoint = {'input_size': 784,
              'output_size': 10,
              'hidden_layers': [each.out_features for each in model.hidden_layers],
              'state_dict': model.state_dict()}

torch.save(checkpoint, 'checkpoint.pth')

Maintenant, le point de contrôle contient toutes les informations nécessaires pour reconstruire le modèle entraîné. Tu peux facilement en faire une fonction si tu le souhaites. De la même manière, nous pouvons écrire une fonction pour charger les points de contrôle. 


In [ ]:
def load_checkpoint(filepath):
    checkpoint = torch.load(filepath)
    model = fc_model.Network(checkpoint['input_size'],
                             checkpoint['output_size'],
                             checkpoint['hidden_layers'])
    model.load_state_dict(checkpoint['state_dict'])
    
    return model

In [ ]:
model = load_checkpoint('checkpoint.pth')
print(model)